<a href="https://colab.research.google.com/github/hoseking86/GTZAN-Assignment2/blob/main/GTZAN_Assignment2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**NB2.1 Accessing the GTZAN Dataset**

This section imports the required Python libraries and confirms that the GTZAN dataset is accessible through Google Drive

In [1]:
#Library imports
import os
from joblib import Parallel, delayed
from tqdm.auto import tqdm
import pandas as pd
import librosa
import numpy as np

In [2]:
#Google drive mounting
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
#Define the path to the GTZAN dataset
dataset_path = "/content/drive/MyDrive/datasets/music_genre"

In [4]:
#Check that the data folder is accessible
!ls "/content/drive/MyDrive/datasets/music_genre"

blues  classical  country  disco  hiphop  jazz	metal  pop  reggae  rock


**NB2.2 Dataset Organisation**

This section organises the GTZAN audio files according to their genres and records the file paths required for feature extraction

In [5]:
#Identify the genre folders
genres = [d.lower() for d in os.listdir(dataset_path)]
genres

['rock',
 'classical',
 'metal',
 'reggae',
 'jazz',
 'country',
 'hiphop',
 'blues',
 'disco',
 'pop']

In [6]:
#Dictionary to store the clips path by genre
audio_files_by_genre = {genre: [] for genre in genres}

#Collect the audio files from each genre folder
for genre in genres:
  genre_path = os.path.join(dataset_path, genre)
  for f in os.listdir(genre_path):
    audio_files_by_genre[genre].append(os.path.join(genre_path, f))

In [7]:
#Display the number of audio files in each genre
for genre, files in audio_files_by_genre.items():
    print(genre, len(files))

rock 100
classical 100
metal 100
reggae 100
jazz 99
country 100
hiphop 100
blues 100
disco 100
pop 100


**NB2.3 Audio Feature Extraction**

This section extracts numerical audio features from the GTZAN recordings using Librosa. The extracted features are combined into a Pandas DataFrame for subsequent validation and machine learning

In [8]:
#Setting the sampling rate
#22kHz is sampling rate of the audio-files
SR = 22050

In [9]:
def extract_features(audio_file, genre, sr=SR):
    """Extract one row of features from a single audio file."""
    try:
        y, sr = librosa.load(audio_file, sr=sr)
    except Exception as e:
        return {'file': audio_file, 'genre': genre, 'error': str(e)}

    # Compute the STFT once and reuse it for every spectral feature
    S = np.abs(librosa.stft(y))                          # magnitude spectrogram
    S_power = S ** 2
    log_mel = librosa.power_to_db(librosa.feature.melspectrogram(S=S_power, sr=sr))

    # Tempo: reuse the log-mel for the onset envelope; skip full beat tracking
    onset_env = librosa.onset.onset_strength(S=log_mel, sr=sr)
    tempo = librosa.feature.tempo(onset_envelope=onset_env, sr=sr)[0]

    centroid = librosa.feature.spectral_centroid(S=S, sr=sr)
    mfcc_means = librosa.feature.mfcc(S=log_mel, n_mfcc=13).mean(axis=1)

    row = {
        'file': audio_file,
        'genre': genre,
        'tempo': float(tempo),
        # Spectral features
        'spectral_centroid_mean': float(centroid.mean()),
        'spectral_centroid_std': float(centroid.std()),
        'spectral_bandwidth_mean': float(librosa.feature.spectral_bandwidth(S=S, sr=sr).mean()),
        'spectral_rolloff_mean': float(librosa.feature.spectral_rolloff(S=S, sr=sr).mean()),
        'spectral_flatness_mean': float(librosa.feature.spectral_flatness(S=S).mean()),
        # Tonal / pitch features
        'chroma_mean': float(librosa.feature.chroma_stft(S=S_power, sr=sr).mean()),
    }
    # Timbral features (MFCCs aggregated per coefficient)
    row.update({f'mfcc_{i}_mean': float(c) for i, c in enumerate(mfcc_means, start=1)})
    return row

# Extracts features from multiple files parallely
def load_dataset(n_jobs=-1):
    jobs = [(f, g) for g, files in audio_files_by_genre.items() for f in files]

    results = Parallel(n_jobs=n_jobs, return_as="generator")(
        delayed(extract_features)(f, g) for f, g in jobs
    )
    rows = list(tqdm(results, total=len(jobs), desc="Extracting features", unit="file"))

    df = pd.DataFrame(rows)
    if 'error' in df.columns:
        failed = df[df['error'].notna()]
        print(f"Skipped {len(failed)} unreadable file(s):")
        for f, err in zip(failed['file'], failed['error']):
            print(f"  {f}: {err}")
        df = df[df['error'].isna()].drop(columns='error').reset_index(drop=True)
    return df

In [10]:
#Extraction of the features and display the first five records
df = load_dataset()
df.head()

Extracting features:   0%|          | 0/999 [00:00<?, ?file/s]

,file,genre,tempo,spectral_centroid_mean,spectral_centroid_std,spectral_bandwidth_mean,spectral_rolloff_mean,spectral_flatness_mean,chroma_mean,mfcc_1_mean,...,mfcc_4_mean,mfcc_5_mean,mfcc_6_mean,mfcc_7_mean,mfcc_8_mean,mfcc_9_mean,mfcc_10_mean,mfcc_11_mean,mfcc_12_mean,mfcc_13_mean
0,/content/drive/MyDrive/datasets/music_genre/ro...,rock,129.199219,2334.519061,336.228937,2208.060155,4819.107544,0.009506,0.392994,-42.662823,...,44.516602,-11.996202,25.576914,-22.668953,28.336857,-19.271856,21.482964,-15.258891,11.748107,-11.575924
1,/content/drive/MyDrive/datasets/music_genre/ro...,rock,99.384014,2975.773811,544.323802,2423.205316,5926.535367,0.021068,0.382693,-95.446480,...,44.241337,-22.469748,29.748888,-21.573271,23.954361,-16.577047,20.471979,-19.630770,13.887503,-12.625713
2,/content/drive/MyDrive/datasets/music_genre/ro...,rock,123.046875,2116.145602,435.115390,2245.391435,4689.983267,0.006532,0.378740,-119.150978,...,58.555363,-12.933258,26.834330,-17.646807,19.371737,-15.605269,13.570684,-16.671040,11.870044,-12.196166
3,/content/drive/MyDrive/datasets/music_genre/ro...,rock,123.046875,1711.169295,1238.594812,2309.187256,3491.701315,0.010117,0.329639,-302.824310,...,25.962639,18.715103,7.566854,7.333934,4.033652,-4.181990,0.879667,0.954835,2.820901,1.406351
4,/content/drive/MyDrive/datasets/music_genre/ro...,rock,112.347147,2511.714306,626.831829,2245.075054,5129.615334,0.006721,0.367960,-202.223434,...,66.306335,-8.287852,24.383793,-16.236677,17.128633,-21.286032,3.962978,-19.760389,0.482673,-4.509064


**NB2.4 Dataset Validation**


This section validates the extracted dataset by checking its dimensions, genre distribution, missing values and duplicate records before splitting it into training and testing datasets

In [11]:
#Checking the extracted dataset for the expected shape
print("Dataset shape:", df.shape)
print("Genre counts:")
print(df["genre"].value_counts())
print("Missing values:", df.isnull().sum().sum())

#Checking for duplications
print("Duplicate file paths:", df["file"].duplicated().sum())

feature_columns = df.drop(columns=["file", "genre"]).columns

print("Duplicate feature rows:",
      df.duplicated(subset=feature_columns).sum())



Dataset shape: (999, 22)
Genre counts:
genre
rock         100
classical    100
metal        100
reggae       100
country      100
hiphop       100
disco        100
blues        100
pop          100
jazz          99
Name: count, dtype: int64
Missing values: 0
Duplicate file paths: 0
Duplicate feature rows: 14


**NB2.5 Training and Testing Dataset Split**

This section divides the validated dataset into 70% training and 30% testing data using stratified sampling to maintain similar genre proportions. Feature scaling is disabled, and a fixed random state is used to support reproducibility

In [12]:
#Import the libraries required for dataset splitting and optional scaling
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


def split_dataset(df, test_size=0.2, target='genre', drop_cols=('file',),
                  stratify=True, scale=True, random_state=42, verbose=True):
    """
    Split the feature DataFrame into train and test sets.

    Parameters
    ----------
    df           : DataFrame returned by load_dataset()
    test_size    : share of data for testing; a fraction (0.2) or a percentage (20)
    target       : name of the label column
    drop_cols    : non-feature columns to exclude (e.g. file paths)
    stratify     : keep the genre proportions the same in train and test
    scale        : standardise features (mean 0, std 1); scaler is fit on train only
    random_state : seed so the split is reproducible
    verbose      : print a summary of the split

    Returns
    -------
    X_train, X_test, y_train, y_test, scaler (None if scale=False)
    """
    if 1 < test_size < 100:          # allow percentages like 20 or 30
        test_size = test_size / 100
    if not 0 < test_size < 1:
        raise ValueError("test_size must be a fraction (0–1) or a percentage (1–99).")

    X = df.drop(columns=[target, *drop_cols], errors='ignore')
    y = df[target]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        stratify=y if stratify else None,
        random_state=random_state,
    )

    scaler = None
    if scale:
        scaler = StandardScaler().fit(X_train)
        X_train = pd.DataFrame(scaler.transform(X_train), columns=X.columns, index=X_train.index)
        X_test = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test.index)

    if verbose:
        print(f"Train: {len(X_train)} samples | Test: {len(X_test)} samples "
              f"| Features: {X.shape[1]} | Test share: {test_size:.0%}")
        print("\nSamples per genre:")
        print(pd.DataFrame({'train': y_train.value_counts(),
                            'test': y_test.value_counts()}).sort_index())

    return X_train, X_test, y_train, y_test, scaler

In [13]:
#Create a reproducible 70/30 stratified split without feature scaling
X_train, X_test, y_train, y_test, _ = split_dataset(df, test_size=30, scale=False)

Train: 699 samples | Test: 300 samples | Features: 20 | Test share: 30%

Samples per genre:
           train  test
genre                 
blues         70    30
classical     70    30
country       70    30
disco         70    30
hiphop        70    30
jazz          69    30
metal         70    30
pop           70    30
reggae        70    30
rock          70    30


**NB2.6 Training and Testing Data Validation**

This section checks the training and testing datasets to confirm their dimensions, missing values, separation and genre distributions. These checks help verify that the datasets have been prepared correctly before model training

In [14]:
#Display the dimensions of the training and testing datasets
print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("Training labels:", y_train.shape)
print("Testing labels:", y_test.shape)

#Check for missing values in the feature datasets
print("Missing training values:", X_train.isnull().sum().sum())
print("Missing testing values:", X_test.isnull().sum().sum())

#Confirm that the training and testing datasets have no shared row indices
print("Shared row indices:",
      len(set(X_train.index) & set(X_test.index)))

Training features: (699, 20)
Testing features: (300, 20)
Training labels: (699,)
Testing labels: (300,)
Missing training values: 0
Missing testing values: 0
Shared row indices: 0


In [15]:
#Preview the training features and labels
X_train.head()

,tempo,spectral_centroid_mean,spectral_centroid_std,spectral_bandwidth_mean,spectral_rolloff_mean,spectral_flatness_mean,chroma_mean,mfcc_1_mean,mfcc_2_mean,mfcc_3_mean,mfcc_4_mean,mfcc_5_mean,mfcc_6_mean,mfcc_7_mean,mfcc_8_mean,mfcc_9_mean,mfcc_10_mean,mfcc_11_mean,mfcc_12_mean,mfcc_13_mean
365,151.999081,2006.849364,524.748730,2120.295215,4323.369243,0.007424,0.390109,-88.075615,109.439316,-25.164967,40.395290,-8.778365,24.714594,-11.525630,18.510265,-8.291809,17.030918,-8.433278,9.799047,-12.903516
547,151.999081,1729.788587,749.306445,2247.607733,3359.521088,0.008072,0.360097,-162.780792,119.890144,-2.907841,20.508930,27.810158,5.490170,6.040869,1.191267,-3.696575,0.352279,4.416835,-0.410248,2.456137
110,123.046875,877.322540,180.951747,1109.262449,1457.712918,0.000354,0.274368,-392.924988,190.735046,-13.309890,8.256798,1.556012,-11.073181,-11.240291,-13.994958,-10.621299,-9.390467,-4.621174,-10.071729,-4.821782
128,117.453835,1285.370259,358.633657,1660.391744,2477.151043,0.001742,0.279380,-360.838684,138.970856,-12.927217,33.778614,7.596159,-1.708812,1.027019,4.072128,3.938996,0.941578,0.516648,4.686481,3.336596
603,107.666016,1963.064381,821.914877,1917.402331,3798.453701,0.008347,0.430894,-138.880600,100.207207,-31.442926,32.449326,-2.751766,13.091758,-14.506294,13.943220,-15.174354,8.177486,-4.925512,3.508959,-7.984948


In [16]:
#Preview the training features and labels
y_train.head()

,genre
365,reggae
547,country
110,classical
128,classical
603,hiphop


In [17]:
#Check that genre proportions are maintained after splitting
print("Training genre distribution:")
print(y_train.value_counts().sort_index())

print("\nTesting genre distribution:")
print(y_test.value_counts().sort_index())

Training genre distribution:
genre
blues        70
classical    70
country      70
disco        70
hiphop       70
jazz         69
metal        70
pop          70
reggae       70
rock         70
Name: count, dtype: int64

Testing genre distribution:
genre
blues        30
classical    30
country      30
disco        30
hiphop       30
jazz         30
metal        30
pop          30
reggae       30
rock         30
Name: count, dtype: int64


**NB2.7 Reproducibility and Project Documentation**

This section creates the project documentation and records the Python dependencies required to reproduce the implementation and testing results

A requirements file is created to document the Python packages required to reproduce the implementation

In [18]:
%%writefile requirements.txt
numpy TBC
pandas TBC
scikit-learn TBC
matplotlib TBC
jupyter TBC

Writing requirements.txt


A README file is created to explain the project, dataset, algorithms and instructions for reproducing the results

In [19]:
%%writefile README.md
# GTZAN Dataset

## Project Overview123456
TBC

TBC

## Dataset
TBC

TBC

## Algorithms
TBC

## Testing and Evaluation
TBC

## Requirements
TBC

    pip install -r requirements.txt

## Running the Notebook
TBC

## Reproducibility
TBC

TBC


Writing README.md
